In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
!pip install --upgrade --quiet langchain langchain-community langchain-groq
!pip install sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.3/114.3 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 109.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 73.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.3/234.3 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires reques

In [2]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 89.6 MB/s eta 0:00:00


In [5]:
import sqlite3
import uuid

from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

In [34]:
import sqlite3
import uuid

from langchain_community.embeddings import HuggingFaceEmbeddings

def download_embeddings():
    model_name = "sentence-transformers/all-MiniLM-L6-v2"

    embeddings = HuggingFaceEmbeddings(                                #converting data into numerical vector chosed this one as it is of low storage , and semantic understanding
        model_name=model_name
    )

    return embeddings

embedding = download_embeddings()
# Set the vectorstore path to your Google Drive
vectorstore_path = "/content/drive/MyDrive/faiss_index(4)/kaggle/working/faiss_index"

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [35]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.load_local(
    vectorstore_path,
    embedding,
    allow_dangerous_deserialization=True
)

In [36]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 5
    }
)

In [37]:
retrieved_docs = retriever.invoke("What is Acne?")
retrieved_docs

[Document(id='fbce068d-91dc-4bf3-8a6f-b88544f7ff64', metadata={'source': '/kaggle/working/depi data/Medical_book.pdf', 'page': 37}, page_content='Nancy J. Nordenson\nAcid reflux see Heartburn\nAcidosis see Respiratory acidosis; Renal\ntubular acidosis; Metabolic acidosis\nAcne\nDefinition\nAcne is a common skin disease characterized by\npimples on the face, chest, and back. It occurs when the\npores of the skin become clogged with oil, dead skin\ncells, and bacteria.\nDescription\nAcne vulgaris, the medical term for common acne, is\nthe most common skin disease. It affects nearly 17 million\npeople in the United States. While acne can arise at any'),
 Document(id='2ae7d67d-170a-45f5-b587-eef9b4fd4da9', metadata={'source': '/kaggle/working/depi data/Medical_book.pdf', 'page': 239}, page_content='used to clear up mild to moderately severe acne.\nIsotretinoin (Accutane) is prescribed only for very\nsevere, disfiguring acne.\nAcne is a skin condition that occurs when pores or\nhair follicl

In [38]:
retrieved_docs = retriever.invoke("What is Fungal infection?")
retrieved_docs

[Document(id='13df7cad-b434-451a-918e-28035bff440b', metadata={'source': '/kaggle/working/depi data/Medical_book.pdf', 'page': 280}, page_content='Fungal infections can either be systemic, meaning\nthat the infection is deep, or topical (dermatophytic),\nmeaning that the infection is superficial and occurs on the\nskin. Additionally, yeast infections can affect the mucous\nmembranes of the body. Fungal infections on the skin are\nusually treated with creams or ointments (topical antifun-\ngal drugs). However, systemic infections, yeast infections\nor topical infections that do not clear up after treatment'),
 Document(id='8e5ca3c1-e5da-4b43-a45b-8c56ea0c89d5', metadata={'source': 'csv'}, page_content='label: Fungal infection | cleaned_text: ive been experiencing a lot of itching which has been accompanied with a rash that appears to be growing worse over time there are also certain areas of skin that are different colours from the rest and ive spotted several lumps that resemble little

In [12]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

In [39]:
system_prompt = (
    "You are a medical assistant for question-answering tasks. "
    "Use only the following retrieved context to answer the question. "
    "If you don't know the answer, say that you don't know. "
    "Do not make up information. "
    "Use a maximum of three sentences and keep the answer concise. "
    "For every answer, include the source page number from the retrieved context "
    "in the format (Page X). If multiple pages are used, include all of them "
    "like (Pages X, Y)."
    "\n\n"
    "Retrieved Context:\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [14]:
!pip install langchain-groq

In [40]:
query = "how to treat acne"
relevant_chunks = retriever.invoke(query)

# Print the relevant chunks (question-answer pairs)
for chunk in relevant_chunks:
    print(chunk.page_content)

salt. Supplementation with herbs such as burdock root
(Arctium lappa ), red clover ( Trifolium pratense ), and
milk thistle (Silybum marianum), and with nutrients such
as essential fatty acids, vitamin B complex, zinc, vitamin
A, and chromium is also recommended. Chinese herbal
remedies used for acne include cnidium seed ( Cnidium
monnieri) and honeysuckle flower ( Lonicera japonica ).
Wholistic physicians or nutritionists can recommend the
proper amounts of these herbs.
Prognosis
• use noncomedogenic makeup and moisturizers
• shampoo often and wear hair off face
• eat a well-balanced diet, avoiding foods that trigger
flare-ups
• unless told otherwise, give dry pimples a limited
amount of sun exposur
• do not pick or squeeze blemishes
• reduce stress
Resources
BOOKS
Balch, James F., and Phyllis A. Balch. “The Disorders: Acne.”
In Prescription for Nutritional Healing, ed. Amy C. Teck-
lenburg, et al. New York: Avery Publishing Group, 1997.
depends upon whether the acne is mild, moderate

In [41]:
query = "Fungal infection"
relevant_chunks = retriever.invoke(query)

# Print the relevant chunks (question-answer pairs)
for chunk in relevant_chunks:
    print(chunk.page_content)

label: Fungal infection | cleaned_text: ive been experiencing a lot of itching which has been accompanied with a rash that appears to be growing worse over time there are also certain areas of skin that are different colours from the rest and ive spotted several lumps that resemble little nodes
label: Fungal infection | cleaned_text: doctor my skin is covered in a very uncomfortable rash along with some odd patches of a different hue my skin also has a few pimples that resemble little knots
label: Fungal infection | cleaned_text: all over my body there has been a severe itching that has been followed by a red bumpy rash my skin also has a few darkened spots and a few tiny nodular breakouts it has been happening for several days and is becoming worse
label: Fungal infection | cleaned_text: ive been experiencing a lot of itching on my skin and sometimes it turns into a rash there are also some strange patches of skin that are a different color than the rest of me and sometimes i get litt

In [42]:
PROMPT_TEMPLATE = """
You are a Medical assistant for question-answering tasks.

Use the following retrieved context to answer the question.
If you don't know the answer, say that you don't know.
Do not make up information.
Keep your answer concise and use a maximum of three sentences.

You MUST include:
- the source name
- the page number

Include them at the end of your answer in this format:
(Source: source_name, Page: X)

If multiple sources are used, list all of them.

Context:
{context}

Question:
{input}
"""

In [43]:
from langchain_core.prompts import ChatPromptTemplate
# Concatenate context text
context_text = "\n\n---\n\n".join([doc.page_content for doc in relevant_chunks])

# Create prompt
prompt_template = ChatPromptTemplate.from_template(PROMPT_TEMPLATE)
prompt = prompt_template.format(
    context=context_text,
    input="what is acne?"
)
print(prompt)

Human: 
You are a Medical assistant for question-answering tasks.

Use the following retrieved context to answer the question.
If you don't know the answer, say that you don't know.
Do not make up information.
Keep your answer concise and use a maximum of three sentences.

You MUST include:
- the source name
- the page number

Include them at the end of your answer in this format:
(Source: source_name, Page: X)

If multiple sources are used, list all of them.

Context:
label: Fungal infection | cleaned_text: ive been experiencing a lot of itching which has been accompanied with a rash that appears to be growing worse over time there are also certain areas of skin that are different colours from the rest and ive spotted several lumps that resemble little nodes

---

label: Fungal infection | cleaned_text: doctor my skin is covered in a very uncomfortable rash along with some odd patches of a different hue my skin also has a few pimples that resemble little knots

---

label: Fungal infe

In [44]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
import os

os.environ["GROQ_API_KEY"] = ""

llm = ChatGroq(
    model_name="llama-3.1-8b-instant"
)

In [47]:
from langchain_core.runnables import RunnablePassthrough

def format_docs(docs):
    return "\n\n".join(
        f"(Source: {doc.metadata.get('source', 'Unknown')}, "
        f"Page: {doc.metadata.get('page', 'Unknown')})\n"
        f"{doc.page_content}"
        for doc in docs
    )

rag_chain = (
    {
        "context": retriever | format_docs,
        "input": RunnablePassthrough()
    }
    | prompt_template
    | llm
)

response = rag_chain.invoke("types of tumors").content
print(response)

There are several types of tumors mentioned in the provided context. These include:

- Gliomas, which are primary brain tumors that arise from glial tissue (Source: /kaggle/working/depi data/Medical_book.pdf, Page: 583)
- Astrocytomas, ependymomas, and mixed gliomas, which are subtypes of gliomas
- Tubular carcinoma, a type of breast cancer (Source: /kaggle/working/depi data/Medical_book.pdf, Page: 606)
- Paget’s disease, a type of adenocarcinoma that can occur in the anal area (Source: /kaggle/working/depi data/Medical_book.pdf, Page: 183)
- Basal cell carcinomas and malignant melanomas, which are types of skin cancer (Source: /kaggle/working/depi data/Medical_book.pdf, Page: 183)
- Benign tumors, which are non-cancerous tumors (Source: /kaggle/working/depi data/Medical_book.pdf, Page: 583)
 
(Sources: /kaggle/working/depi data/Medical_book.pdf, Pages: 583, 606, 183)


In [48]:
import time

def timed_hybrid(query, k=5):
    start = time.time()

    # correct way to call retriever
    docs = retriever.invoke(query)

    end = time.time()

    return {
        "docs": docs[:k]
    }, (end - start) * 1000  # milliseconds

In [49]:
def recall_at_k(result, expected_condition):
    """
    Checks if the correct condition exists
    inside top-k retrieved candidates.
    """
    for h in result["candidates"]:
        if h["metadata"]["condition"] == expected_condition:
            return 1
    return 0

In [50]:
def evaluate_system(eval_data, k=5):
    total = len(eval_data)

    hits = 0
    recall_hits = 0
    total_latency = 0

    for item in eval_data:
        query = item["query"]
        expected = item["expected_condition"]  # CHANGED (medical meaning)

        result, latency = timed_hybrid(query, k=k)
        total_latency += latency

        retrieved_docs = result["docs"]  # assume list of retrieved chunks

        # -------------------------
        # HIT RATE
        # -------------------------
        if retrieved_docs:
            hits += 1

        # -------------------------
        # RECALL@K (keyword-based simple version)
        # -------------------------
        retrieved_text = " ".join([d.page_content.lower() for d in retrieved_docs])
        expected_words = set(expected.lower().split())

        overlap = sum(1 for w in expected_words if w in retrieved_text)

        recall_hits += overlap / max(len(expected_words), 1)

    return {
        "Total Queries": total,
        "Hit Rate@K": hits / total,
        "Recall@K": recall_hits / total,
        "Average Latency (ms)": total_latency / total
    }

In [51]:
base_queries = [
    {
        "query": "What are the common symptoms of eczema?",
        "expected_condition": "itching, dry skin, redness, inflamed or scaly patches"
    },
    {
        "query": "How is anemia treated?",
        "expected_condition": "iron supplements, dietary changes, treating underlying cause"
    },
    {
        "query": "What causes migraine headaches?",
        "expected_condition": "triggers such as stress, hormonal changes, certain foods, lack of sleep"
    },
    {
        "query": "What are the complications of untreated asthma?",
        "expected_condition": "breathing difficulties, frequent attacks, reduced lung function, hospitalization"
    },
    {
        "query": "How is rheumatoid arthritis diagnosed?",
        "expected_condition": "physical examination, blood tests, imaging studies like X-rays or MRI"
    },
    {
        "query": "What are the symptoms of hypothyroidism?",
        "expected_condition": "fatigue, weight gain, cold intolerance, dry skin, slow heart rate"
    },
    {
        "query": "How is chronic kidney disease managed?",
        "expected_condition": "blood pressure control, dietary restrictions, medications, dialysis if severe"
    },
    {
        "query": "What causes osteoporosis?",
        "expected_condition": "aging, hormonal changes, calcium deficiency, lack of physical activity"
    },
    {
        "query": "How is influenza diagnosed?",
        "expected_condition": "clinical symptoms, rapid flu test, PCR testing"
    },
    {
        "query": "What are the warning signs of a stroke?",
        "expected_condition": "sudden weakness, facial drooping, speech difficulty, vision problems, dizziness"
    }
]

# Expand to 100 queries by reusing + slight variation
evaluation_queries = []

topics = [
    "eczema", "anemia", "migraine", "asthma", "rheumatoid arthritis",
    "hypothyroidism", "kidney disease", "osteoporosis", "influenza", "stroke"
]

templates = [
    ("What are the symptoms of {}", "symptoms vary depending on condition"),
    ("What causes {}?", "causes include genetic, environmental, and lifestyle factors"),
    ("How is {} diagnosed?", "diagnosis includes clinical evaluation and medical tests"),
    ("How is {} treated?", "treatment includes medications and lifestyle changes"),
    ("What are complications of {}?", "complications depend on severity and untreated progression")
]

for i in range(100):
    topic = topics[i % len(topics)]
    q_template, expected = templates[i % len(templates)]

    evaluation_queries.append({
        "query": q_template.format(topic),
        "expected_condition": expected
    })

evaluation_queries = evaluation_queries[:100]

In [52]:
metrics = evaluate_system(evaluation_queries, k=10)
metrics

{'Total Queries': 100,
 'Hit Rate@K': 1.0,
 'Recall@K': 0.40333333333333343,
 'Average Latency (ms)': 6.8293046951293945}

In [53]:
def answer_question_rag(user_input):
    # This will invoke the RAG chain you have already set up
    return rag_chain.invoke(user_input).content

In [ ]:
def get_prompt(user_input, context):
    prompt = f"""
You are a Medical assistant for question-answering tasks.

Use the following retrieved context to answer the question.
If you don't know the answer, say that you don't know.
Do not make up information.
Keep your answer concise and use a maximum of three sentences.

Context:
{context}

Question:
{user_input}
"""
    return llm.invoke(prompt).content

In [60]:
def answer_question(user_input):

    # 1. Retrieve relevant docs
    docs = retriever.invoke(user_input)

    # 2. Build context with source + page number (+1)
    context = "\n\n".join([
        f"(Source: {doc.metadata.get('source', 'Unknown')}, "
        f"Page: {doc.metadata.get('page', 0) + 1})\n"
        f"{doc.page_content}"
        for doc in docs
    ])

    # 3. Create prompt
    prompt = f"""
You are a Medical assistant for question-answering tasks.

Use the following retrieved context to answer the question.
If you don't know the answer, say that you don't know.
Do not make up information.
Keep your answer concise (max 3 sentences).

You MUST include the source name and page number in your answer.
Use this format:
(Source: source_name, Page: X)

Context:
{context}

Question:
{user_input}
"""

    # 4. Get model response
    answer = llm.invoke(prompt).content

    # 5. Return only the answer
    return answer

In [55]:
pip install --upgrade gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.7/19.7 MB 96.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.4 MB/s eta 0:00:00
  Attempting uninstall: gradio-client
    Found existing installation: gradio_client 1.14.0
    Uninstalling gradio_client-1.14.0:
      Successfully uninstalled gradio_client-1.14.0
  Attempting uninstall: gradio
    Found existing installation: gradio 5.50.0
    Uninstalling gradio-5.50.0:
      Successfully uninstalled gradio-5.50.0


In [61]:
import gradio as gr

interface = gr.Interface(
    fn=answer_question,
    inputs=gr.Textbox(label="Enter Your Medical Question"),
    outputs=[
        gr.Textbox(label="Answer")

    ]
)

interface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://98de9d9aed7b6a533a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
